# Cats vs Dogs baseline training

Thin Kaggle orchestration only: all preprocessing, model, training, evaluation, MLflow, and artifact logic comes from the repository source snapshot.

In [ ]:
import os, shutil, subprocess, zipfile
from pathlib import Path

input_root = Path('/kaggle/input')
print('Attached inputs:', [path.name for path in input_root.iterdir()])
source_zips = list(input_root.rglob('mlops_source.zip'))
project_root = Path('/kaggle/working/mlops_binary_class')
project_root.mkdir(parents=True, exist_ok=True)
if source_zips:
    with zipfile.ZipFile(source_zips[0]) as archive:
        archive.extractall(project_root)
else:
    extracted_configs = list(input_root.rglob('configs/base.yaml'))
    if not extracted_configs:
        raise RuntimeError(f'Repository source snapshot is not attached below {input_root}')
    shutil.copytree(extracted_configs[0].parents[1], project_root, dirs_exist_ok=True)
os.chdir(project_root)
print('Repository source ready:', project_root)

In [ ]:
# Kaggle supplies PyTorch/torchvision/Pillow/NumPy/scikit-learn. Install MLflow only if absent.
try:
    import mlflow
except ImportError:
    subprocess.run(['python', '-m', 'pip', 'install', '-q', 'mlflow==3.15.2'], check=True)
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
dataset_input_root = Path('/kaggle/input')
candidates = [p for p in dataset_input_root.rglob('*') if p.is_dir() and (p / 'Cat').is_dir() and (p / 'Dog').is_dir()]
if not candidates:
    raise RuntimeError(f'Could not find Cat/Dog directories below {dataset_input_root}')
raw_data_dir = min(candidates, key=lambda p: len(p.parts))
processed_dir = Path('/kaggle/working/PetImages224')
print('Raw dataset:', raw_data_dir)
subprocess.run([
    'python', '-m', 'scripts.materialize_processed_dataset',
    '--source-dir', str(raw_data_dir),
    '--manifests-dir', 'data/manifests/baseline_50',
    '--output-dir', str(processed_dir),
    '--workers', '8',
], check=True)

In [ ]:
subprocess.run([
    'python', '-m', 'src.training.train',
    '--config', 'configs/base.yaml',
    '--processed-dir', str(processed_dir),
    '--device', 'cuda',
    '--run-name', 'simple-cnn-baseline-50pct',
    '--execution-environment', 'kaggle',
    '--promote-to-production',
], check=True)